In [3]:
import pandas as pd

# --- INPUTS ---
initial_corpus = 24.14e5 # ₹24.14 lakh
initial_sip = 35000 # ₹35,000/month
sip_increase_rate = 0.10 # 10% annual SIP increase
annual_return = 0.12 # 12% p.a.
months_per_year = 12
target = 10e7 # ₹10 crore

# --- CALCULATION ---
r = annual_return / 12 # monthly rate
year, age = 0, 34
value = initial_corpus
sip = initial_sip

records = []

while value < target and year < 50:
 year += 1
 age += 1

# add SIP each month and compound
for m in range(12):
 value = value * (1 + r) + sip

# total invested till now
total_invested = initial_corpus + sum(
[initial_sip * ((1 + sip_increase_rate) ** i) * months_per_year for i in range(year)]
)

records.append({
"Year": year,
"Age": age,
"Monthly SIP (₹)": round(sip, 0),
"Total Invested (₹ Lakh)": round(total_invested / 1e5, 2),
"Projected Value (₹ Cr)": round(value / 1e7, 2)
})

sip *= (1 + sip_increase_rate) # increase SIP for next year

df = pd.DataFrame(records)
print(df)


   Year  Age  Monthly SIP (₹)  Total Invested (₹ Lakh)  Projected Value (₹ Cr)
0    50   84            35000                  4912.56                    0.32


In [16]:
import yahoofinance as yf
stklist = ['ABCAPITAL',
'ARE&M',
'AXISBANK',
'DRREDDY',
'FINCABLES',
'HCLTECH',
'HDBFS',
'IDFCFIRSTB',
'INDUSINDBK',
'INFY',
'ITBEES',
'ITC',
'ITCHOTELS',
'JYOTHYLAB',
'KTKBANK',
'MANAPPURAM',
'MOTILALOFS',
'MUTHOOTFIN',
'NATCOPHARM',
'NEXT50IETF',
'NIFTYBEES',
'PETRONET',
'SOUTHBANK',
'SPANDANA',
'TATACAP',
'TATACHEM',
'TATASTEEL',
'TATATECH',
'TCS',
'TMB',
'TMPV',
'WIPRO',
'YESBANK',
'ZYDUSLIFE']
df = []
for stock in stklist:
    hp = yf.History(stock, period="1mo", interval="1d")
    df.append(hp.get_prices())


AttributeError: module 'yahoofinance' has no attribute 'History'

In [5]:
# ...existing code...
from yahoofinance import HistoricalPrices
import pandas as pd

stklist = [
 'ABCAPITAL','ARE&M','AXISBANK','DRREDDY.NS','FINCABLES','HCLTECH','HDBFS',
 'IDFCFIRSTB','INDUSINDBK','INFY','ITBEES','ITC','ITCHOTELS','JYOTHYLAB',
 'KTKBANK','MANAPPURAM','MOTILALOFS','MUTHOOTFIN','NATCOPHARM','NEXT50IETF',
 'NIFTYBEES','PETRONET','SOUTHBANK','SPANDANA','TATACAP','TATACHEM',
 'TATASTEEL','TATATECH','TCS','TMB','TMPV','WIPRO','YESBANK','ZYDUSLIFE'
]

# If these are NSE tickers, append .NS (remove or adjust if not needed)
tickers = [t if '.' in t else t + '.NS' for t in stklist]

all_data = {}
for stock in tickers:
    try:
        hp = HistoricalPrices(stock, '2025-11-06', '2025-11-06')
        df = hp.get_prices()
        if df is None or df.empty:
            print(f"No data for {stock}")
            continue
        df = df.copy()
        df['Ticker'] = stock
        all_data[stock] = df
        print(f"Fetched {len(df)} rows for {stock}")
    except Exception as e:
        print(f"Error fetching {stock}: {e}")

if all_data:
    combined = pd.concat(all_data.values(), keys=all_data.keys(), names=['Ticker','Row'])
    combined.to_csv('prices_2025-11-06.csv')
    display(combined)
else:
    print("No data fetched.")
# ...existing code...

Error fetching ABCAPITAL.NS: Cookie not found
Error fetching ARE&M.NS: Cookie not found
Error fetching AXISBANK.NS: Cookie not found
Error fetching DRREDDY.NS: Cookie not found
Error fetching FINCABLES.NS: Cookie not found
Error fetching HCLTECH.NS: Cookie not found
Error fetching HDBFS.NS: Cookie not found
Error fetching IDFCFIRSTB.NS: Cookie not found
Error fetching INDUSINDBK.NS: Cookie not found
Error fetching INFY.NS: Cookie not found
Error fetching ITBEES.NS: Cookie not found
Error fetching ITC.NS: Cookie not found
Error fetching ITCHOTELS.NS: Cookie not found
Error fetching JYOTHYLAB.NS: Cookie not found
Error fetching KTKBANK.NS: Cookie not found
Error fetching MANAPPURAM.NS: Cookie not found
Error fetching MOTILALOFS.NS: Cookie not found
Error fetching MUTHOOTFIN.NS: Cookie not found
Error fetching NATCOPHARM.NS: Cookie not found
Error fetching NEXT50IETF.NS: Cookie not found
Error fetching NIFTYBEES.NS: Cookie not found
Error fetching PETRONET.NS: Cookie not found
Error fetch

In [6]:
# ...existing code...
# Use yfinance to avoid the "Cookie not found" issue
!pip install --quiet yfinance

import yfinance as yf
import pandas as pd

# reuse your stklist variable from the notebook
tickers = [t if '.' in t else t + '.NS' for t in stklist]

start = '2025-11-06'
end = '2025-11-07'  # yfinance end is exclusive, so use next day to include the 6th

# Download all tickers in one call (returns MultiIndex columns when multiple tickers)
data = yf.download(tickers, start=start, end=end, group_by='ticker', threads=True, progress=False)

all_data = {}
if isinstance(data.columns, pd.MultiIndex):
    for ticker in tickers:
        try:
            df = data[ticker].dropna(how='all')
            if df.empty:
                print(f"No data for {ticker}")
                continue
            df = df.copy()
            df['Ticker'] = ticker
            all_data[ticker] = df
            print(f"Fetched {len(df)} rows for {ticker}")
        except Exception as e:
            print(f"Error processing {ticker}: {e}")
else:
    # single-ticker result
    df = data.dropna(how='all')
    if not df.empty:
        df['Ticker'] = tickers[0]
        all_data[tickers[0]] = df

if all_data:
    combined = pd.concat(all_data.values(), keys=all_data.keys(), names=['Ticker','Row'])
    combined.to_csv('prices_2025-11-06.csv')
    display(combined)
else:
    print("No data fetched.")
# ...existing code...


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
Failed to get ticker 'ZYDUSLIFE.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'TATACAP.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'NIFTYBEES.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'IDFCFIRSTB.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'HCLTECH.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'INDUSINDBK.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'INFY.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'DRREDDY.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'SOUTHBANK.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'WIPRO.NS' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'SPANDANA

No data for ABCAPITAL.NS
No data for ARE&M.NS
No data for AXISBANK.NS
No data for DRREDDY.NS
No data for FINCABLES.NS
No data for HCLTECH.NS
No data for HDBFS.NS
No data for IDFCFIRSTB.NS
No data for INDUSINDBK.NS
No data for INFY.NS
No data for ITBEES.NS
No data for ITC.NS
No data for ITCHOTELS.NS
No data for JYOTHYLAB.NS
No data for KTKBANK.NS
No data for MANAPPURAM.NS
No data for MOTILALOFS.NS
No data for MUTHOOTFIN.NS
No data for NATCOPHARM.NS
No data for NEXT50IETF.NS
No data for NIFTYBEES.NS
No data for PETRONET.NS
No data for SOUTHBANK.NS
No data for SPANDANA.NS
No data for TATACAP.NS
No data for TATACHEM.NS
No data for TATASTEEL.NS
No data for TATATECH.NS
No data for TCS.NS
No data for TMB.NS
No data for TMPV.NS
No data for WIPRO.NS
No data for YESBANK.NS
No data for ZYDUSLIFE.NS
No data fetched.


In [7]:
# ...existing code...
import time
import logging
import yfinance as yf
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

tickers = [t if '.' in t else t + '.NS' for t in stklist]
start = '2025-11-06'
end = '2025-11-07'  # end is exclusive

def fetch_one(ticker, retries=3, backoff=1.0):
    for attempt in range(1, retries+1):
        try:
            # use Ticker.history (more reliable for single symbols) and avoid threaded download
            df = yf.Ticker(ticker).history(start=start, end=end)
            if df is None or df.empty:
                logging.info(f"No data for {ticker}")
                return None
            df = df.copy()
            df['Ticker'] = ticker
            return df
        except Exception as e:
            logging.warning(f"Attempt {attempt} failed for {ticker}: {e}")
            time.sleep(backoff * attempt)
    logging.error(f"Failed to get ticker '{ticker}' after {retries} attempts")
    return None

all_data = {}
for t in tickers:
    df = fetch_one(t)
    if df is not None:
        all_data[t] = df
        logging.info(f"Fetched {len(df)} rows for {t}")

if all_data:
    combined = pd.concat(all_data.values(), keys=all_data.keys(), names=['Ticker','Row'])
    combined.to_csv('prices_2025-11-06.csv')
    display(combined)
else:
    print("No data fetched.")
# ...existing code...

ERROR: Failed to get ticker 'ABCAPITAL.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $ABCAPITAL.NS: possibly delisted; no timezone found
INFO: No data for ABCAPITAL.NS
ERROR: Failed to get ticker 'ARE&M.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $ARE&M.NS: possibly delisted; no timezone found
INFO: No data for ARE&M.NS
ERROR: Failed to get ticker 'AXISBANK.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $AXISBANK.NS: possibly delisted; no timezone found
INFO: No data for AXISBANK.NS
ERROR: Failed to get ticker 'DRREDDY.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $DRREDDY.NS: possibly delisted; no timezone found
INFO: No data for DRREDDY.NS
ERROR: Failed to get ticker 'FINCABLES.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $FINCABLES.NS: possibly delisted; no timezone found
INFO: No data for FINCABLES.NS
ERROR: Failed to get ticker 'HCLTECH.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: $HCLTECH

No data fetched.


In [8]:
# ...existing code...
import time
import logging
import yfinance as yf
import pandas as pd
from json import JSONDecodeError

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# prepare tickers from your stklist
tickers = [t if '.' in t else t + '.NS' for t in stklist]
start = '2025-11-06'
end = '2025-11-07'  # end is exclusive

def validate_ticker(ticker):
    try:
        info = yf.Ticker(ticker).info
        if not info:
            return False, "empty info (possibly delisted or invalid)"
        # basic sanity checks
        if 'symbol' not in info:
            return False, f"no symbol in info, keys: {list(info.keys())[:5]}"
        return True, info
    except (JSONDecodeError, ValueError, TypeError) as e:
        return False, f"JSON/parse error: {e}"
    except Exception as e:
        return False, f"info fetch error: {e}"

def fetch_one(ticker, retries=3, backoff=1.0):
    for attempt in range(1, retries+1):
        try:
            df = yf.Ticker(ticker).history(start=start, end=end)
            if df is None or df.empty:
                logging.info(f"No data for {ticker}")
                return None
            df = df.copy()
            df['Ticker'] = ticker
            return df
        except Exception as e:
            logging.warning(f"Attempt {attempt} failed for {ticker}: {e}")
            time.sleep(backoff * attempt)
    logging.error(f"Failed to get ticker '{ticker}' after {retries} attempts")
    return None

all_data = {}
for t in tickers:
    valid, reason = validate_ticker(t)
    if not valid:
        logging.error(f"Skipping {t}: {reason}")
        continue
    df = fetch_one(t)
    if df is not None:
        all_data[t] = df
        logging.info(f"Fetched {len(df)} rows for {t}")

if all_data:
    combined = pd.concat(all_data.values(), keys=all_data.keys(), names=['Ticker','Row'])
    combined.to_csv('prices_2025-11-06.csv')
    display(combined)
else:
    print("No data fetched.")
# ...existing code...

ERROR: 429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ABCAPITAL.NS?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ABCAPITAL.NS&crumb=Edge%3A+Too+Many+Requests
ERROR: Skipping ABCAPITAL.NS: JSON/parse error: Expecting value: line 1 column 1 (char 0)
ERROR: 429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/ARE&M.NS?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=ARE%26M.NS&crumb=Edge%3A+Too+Many+Requests
ERROR: Skipping ARE&M.NS: JSON/parse error: Expecting value: line 1 column 1 (char 0)
ERROR: 429 Client Error: Too Many Requests for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/AXISBANK.NS?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&cors

KeyboardInterrupt: 

In [9]:
# ...existing code...
import time
import logging
import yfinance as yf
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

tickers = [t if '.' in t else t + '.NS' for t in stklist]
start = '2025-11-06'
end = '2025-11-07'  # end is exclusive

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def fetch_single(ticker, retries=3):
    backoff = 2
    for attempt in range(1, retries+1):
        try:
            df = yf.Ticker(ticker).history(start=start, end=end)
            if df is None or df.empty:
                logging.info(f"No data for {ticker}")
                return None
            df = df.copy(); df['Ticker'] = ticker
            return df
        except Exception as e:
            msg = str(e)
            logging.warning(f"Attempt {attempt} failed for {ticker}: {msg}")
            if '429' in msg or 'Too Many Requests' in msg:
                time.sleep(10 * attempt)  # longer sleep on rate-limit
            else:
                time.sleep(backoff)
            backoff *= 2
    logging.error(f"Failed to get ticker '{ticker}' after {retries} attempts")
    return None

all_data = {}
batch_size = 10  # reduce to avoid rate limits; tune as needed
for batch in chunks(tickers, batch_size):
    attempts = 0
    while attempts < 3:
        try:
            # use yf.download in batches, threads=False to reduce concurrent hits
            data = yf.download(batch, start=start, end=end, group_by='ticker', threads=False, progress=False)
            break
        except Exception as e:
            attempts += 1
            msg = str(e)
            logging.warning(f"Batch download attempt {attempts} failed: {msg}")
            if '429' in msg or 'Too Many Requests' in msg:
                sleep_for = 30 * attempts
                logging.info(f"Rate limited — sleeping {sleep_for}s before retry")
                time.sleep(sleep_for)
            else:
                time.sleep(5 * attempts)
    else:
        # batch failed after retries — fallback to single-ticker fetches
        logging.error(f"Batch {batch} failed after retries, falling back to singles")
        for t in batch:
            df = fetch_single(t)
            if df is not None:
                all_data[t] = df
        time.sleep(2)
        continue

    # process batch result
    if isinstance(data.columns, pd.MultiIndex):
        for t in batch:
            try:
                df = data[t].dropna(how='all')
                if df.empty:
                    logging.info(f"No data for {t}")
                    continue
                df = df.copy(); df['Ticker'] = t
                all_data[t] = df
                logging.info(f"Fetched {len(df)} rows for {t}")
            except Exception as e:
                logging.warning(f"Error processing {t}: {e}")
    else:
        # single-ticker result when batch has one symbol
        df = data.dropna(how='all')
        if not df.empty:
            single_t = batch[0]
            df = df.copy(); df['Ticker'] = single_t
            all_data[single_t] = df
            logging.info(f"Fetched {len(df)} rows for {single_t}")

    # polite pause between batches
    time.sleep(2)

if all_data:
    combined = pd.concat(all_data.values(), keys=all_data.keys(), names=['Ticker','Row'])
    combined.to_csv('prices_2025-11-06.csv')
    display(combined)
else:
    print("No data fetched.")
# ...existing code...

ERROR: Failed to get ticker 'IDFCFIRSTB.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'FINCABLES.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'ARE&M.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'ABCAPITAL.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'INDUSINDBK.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'DRREDDY.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'HDBFS.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'INFY.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'HCLTECH.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: Failed to get ticker 'AXISBANK.NS' reason: Expecting value: line 1 column 1 (char 0)
ERROR: 
10 Failed downloads:
ERROR: ['IDFCFIRSTB.NS', 'FINCABLES.NS', 'ARE&M.NS', 'ABC

No data fetched.


In [13]:
import yfinance as yf
tickers = yf.Tickers('DR.REDDY')
tickers.history(period='1d')

ERROR: Failed to get ticker 'DR.REDDY' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 1 completed
ERROR: 
1 Failed download:
ERROR: ['DR.REDDY']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


KeyError: 'DR.REDDY'